<a href="https://colab.research.google.com/github/Poojarautela03/ABTALKS/blob/main/Day_21_What%20Makes%20AI%20Products%20Actually%20Good/quality_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 21 — What Makes AI Products Actually Good 🧪

A structured product-quality audit of the Day 20 AI knowledge assistant. This notebook rebuilds the Day 20 pipeline (chunking, FAISS search, metadata filtering, grounding prompt) and then tests it against five product quality dimensions, not just "does it work."

> **Provider note:** uses **Google Gemini** (`gemini-embedding-001` + `gemini-flash-lite-latest`) instead of OpenAI, due to OpenAI API quota limits hit earlier in the challenge.

In [1]:
!pip install -q google-generativeai faiss-cpu numpy langchain-text-splitters

import os, time, json, statistics
from datetime import datetime
from typing import List, Dict, Any, Optional

import numpy as np
import faiss
import google.generativeai as genai
from langchain_text_splitters import RecursiveCharacterTextSplitter
from google.colab import userdata

genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
chat_model = genai.GenerativeModel('gemini-flash-lite-latest')
EMBEDDING_MODEL = "models/gemini-embedding-001"
LOW_CONFIDENCE_THRESHOLD = 0.3

print("Setup done.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 28.5 MB/s eta 0:00:00


/usr/local/lib/python3.13/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Setup done.


## Rebuild the Day 20 pipeline (same knowledge base, chunking, FAISS index, grounding prompt)

In [2]:
class Document:
    def __init__(self, doc_id, title, text, category, date, document_type):
        self.doc_id = doc_id
        self.title = title
        self.text = text
        self.category = category
        self.date = date
        self.document_type = document_type


class Chunk:
    def __init__(self, chunk_id, text, source_doc):
        self.chunk_id = chunk_id
        self.text = text
        self.source_doc = source_doc

    def to_dict(self):
        return {
            "chunk_id": self.chunk_id, "text": self.text,
            "source_id": self.source_doc.doc_id, "source_title": self.source_doc.title,
            "category": self.source_doc.category, "date": self.source_doc.date,
            "document_type": self.source_doc.document_type,
        }


DOCUMENTS = [
    Document("doc_company_history", "Company History",
        "Nimbus Robotics was founded in 2031 by engineer Priya Kalathil in Pune, India. "
        "The company began as a three-person team working out of a small office near "
        "Hinjewadi, focused on solving warehouse automation problems for mid-sized logistics "
        "firms. Within its first two years, Nimbus Robotics grew to employ 212 people across "
        "three offices: Pune, Bengaluru, and Singapore. Priya Kalathil remains CEO as of 2034.",
        "company_background", "2031-03-15", "reference"),
    Document("doc_leadership", "Leadership Team",
        "The company's CTO, Rohan Mehta, previously led robotics research at a university lab "
        "for eight years before joining Nimbus Robotics in 2031 as a founding engineer. Rohan "
        "holds a doctorate in mechanical engineering. Ananya Desai joined as VP of Operations "
        "in 2032 after a decade in supply chain management.",
        "company_background", "2031-04-01", "reference"),
    Document("doc_aster7_specs", "Aster-7 Product Specifications",
        "Nimbus Robotics' flagship product is the Aster-7, a warehouse picking robot with a "
        "99.2% accuracy rate. The Aster-7 uses a proprietary gripper called FlexGrip, which "
        "adjusts pressure using 12 micro-sensors per finger. The Aster-7's battery lasts 14 "
        "hours on a single charge and recharges fully in 40 minutes.",
        "product", "2033-01-10", "technical"),
    Document("doc_aster7_deployment", "Aster-7 Market Deployment",
        "The Aster-7 has been deployed in over 60 warehouses across South and Southeast Asia "
        "since its commercial launch in 2033. Customer feedback has been generally positive, "
        "with the most common request being for improved battery swap mechanisms.",
        "market", "2033-08-15", "marketing"),
    Document("doc_aster8_roadmap", "Aster-8 Product Roadmap",
        "Nimbus Robotics' next product, the Aster-8, is scheduled for release in early 2035. "
        "The Aster-8 will support hot-swappable battery packs, reducing downtime from 40 "
        "minutes to under 5 minutes, and an upgraded FlexGrip with 18 micro-sensors per finger.",
        "product", "2034-09-01", "marketing"),
    Document("doc_finance", "Financial Performance",
        "Nimbus Robotics reported revenue of 340 million rupees in fiscal year 2033, an "
        "increase of approximately 65% over the prior year. The company has not yet reached "
        "profitability due to continued R&D investment in the Aster-8 program.",
        "finance", "2033-12-31", "reference"),
    Document("doc_competition", "Competitive Landscape",
        "Nimbus Robotics' main competitor is Solace Automation, founded a year earlier in "
        "2030. Solace Automation focuses on a broader range of warehouse robotics, including "
        "picking robots and autonomous forklifts.",
        "market", "2032-06-01", "marketing"),
]

print(f"Loaded {len(DOCUMENTS)} documents.")

Loaded 7 documents.


In [3]:
GROUNDING_SYSTEM_PROMPT = """You are a strict, grounded knowledge assistant. You must ONLY
answer using information explicitly stated in the provided context below.

Rules:
1. If the answer is not directly supported by the context, respond exactly with:
   "I cannot answer this based on the available knowledge base."
2. Do NOT use any external knowledge, training data, or assumptions to fill gaps.
3. Do NOT guess, infer, or extrapolate beyond what is explicitly written in the context.
4. Cite which source(s) you used by referencing their [source_id] tags inline.

CONTEXT:
{context}

QUESTION:
{question}

ANSWER (include [source_id] citations inline where you use a fact):"""


def chunk_documents(documents, chunk_size=500, chunk_overlap=100):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = []
    for doc in documents:
        pieces = splitter.split_text(doc.text)
        for i, piece in enumerate(pieces):
            chunks.append(Chunk(f"{doc.doc_id}::chunk{i}", piece, doc))
    return chunks


def get_embeddings(texts, max_retries=3):
    embeddings = []
    for text in texts:
        for attempt in range(max_retries):
            try:
                result = genai.embed_content(model=EMBEDDING_MODEL, content=text)
                embeddings.append(result["embedding"])
                break
            except Exception as e:
                if attempt < max_retries - 1:
                    time.sleep(5)
                else:
                    raise RuntimeError(f"Embedding failed: {e}")
        time.sleep(1)
    return embeddings


class RAGIndex:
    def __init__(self, chunks):
        self.chunks = chunks
        texts = [c.text for c in chunks]
        vectors = np.array(get_embeddings(texts), dtype=np.float32)
        self.dimension = vectors.shape[1]
        self.index = faiss.IndexFlatL2(self.dimension)
        self.index.add(vectors)

    def search(self, query, top_k=3, filters=None, search_pool=10):
        query_vector = np.array(get_embeddings([query]), dtype=np.float32)
        distances, indices = self.index.search(query_vector, search_pool)
        results = []
        for dist, idx in zip(distances[0], indices[0]):
            if idx == -1:
                continue
            chunk = self.chunks[idx]
            entry = chunk.to_dict()
            entry["distance"] = float(dist)
            entry["similarity"] = float(1 / (1 + dist))
            results.append(entry)
            if len(results) >= top_k:
                break
        return results


print("Building index (embeds every chunk, ~30-60s)...")
_chunks = chunk_documents(DOCUMENTS)
_index = RAGIndex(_chunks)
print(f"Index built with {len(_chunks)} chunks.")

Building index (embeds every chunk, ~30-60s)...
Index built with 7 chunks.


In [4]:
def generate_answer_timed(query, index, top_k=3):
    """Same as Day 20's generate_answer, but returns per-component timing
    so we can measure which stage of the pipeline is slowest."""
    timings = {}

    t0 = time.time()
    retrieved = index.search(query, top_k=top_k)
    timings["retrieval_seconds"] = time.time() - t0

    if not retrieved:
        return {
            "answer": "I cannot answer this based on the available knowledge base.",
            "sources": [], "low_confidence": True, "top_similarity": 0.0,
            "timings": timings, "total_seconds": timings["retrieval_seconds"]
        }

    t1 = time.time()
    context_lines = [f"[{r['source_id']}] ({r['source_title']}): {r['text']}" for r in retrieved]
    context = "\n".join(context_lines)
    prompt = GROUNDING_SYSTEM_PROMPT.format(context=context, question=query)
    timings["prompt_build_seconds"] = time.time() - t1

    t2 = time.time()
    response = chat_model.generate_content(
        prompt, generation_config=genai.types.GenerationConfig(temperature=0)
    )
    timings["generation_seconds"] = time.time() - t2
    answer_text = response.text.strip()

    best_similarity = max(r["similarity"] for r in retrieved)
    low_confidence = best_similarity < LOW_CONFIDENCE_THRESHOLD
    if low_confidence:
        answer_text += ("\n\n\u26a0\ufe0f Low confidence: the retrieved information may not be "
                         "well-supported for this question.")

    sources = [{"source_id": r["source_id"], "title": r["source_title"],
                "chunk_id": r["chunk_id"], "similarity": round(r["similarity"], 4)} for r in retrieved]

    timings["total_seconds"] = sum(timings.values())

    return {"answer": answer_text, "sources": sources, "low_confidence": low_confidence,
            "top_similarity": round(best_similarity, 4), "timings": timings,
            "total_seconds": timings["total_seconds"]}

print("Timed pipeline function ready.")

Timed pipeline function ready.


## Five product quality dimensions — definitions

1. **Accuracy** — the answer correctly reflects what the knowledge base actually states, with no fabricated or omitted facts.
2. **Latency** — the time between a user submitting a query and receiving a complete response, end to end.
3. **Reliability** — the system produces a valid, well-formed response consistently across repeated and varied queries, without crashing or returning malformed output.
4. **Transparency** — every answer clearly shows the user where the information came from, in a way a non-technical person could understand and verify.
5. **Graceful degradation** — when the system can't answer confidently, it says so honestly instead of guessing or failing silently.

## Dimension 1 & 2: Accuracy + Latency — 10 queries, timed

In [5]:
accuracy_test_queries = [
    "Who founded Nimbus Robotics?",
    "What is the Aster-7's battery life?",
    "How much revenue did Nimbus Robotics report in fiscal year 2033?",
    "Who is Nimbus Robotics' main competitor?",
    "When is the Aster-8 scheduled for release?",
    "What is the Aster-7's gripper called?",
    "How many people does Nimbus Robotics employ?",
    "What background does the CTO have?",
    "Where has the Aster-7 been deployed?",
    "What is Solace Automation's product focus?",
]

accuracy_latency_results = []
for i, q in enumerate(accuracy_test_queries, start=1):
    print(f"[{i}/10] {q}")
    result = generate_answer_timed(q, _index)
    print(f"  Answer: {result['answer'][:150]}")
    print(f"  Total time: {result['total_seconds']:.2f}s | "
          f"retrieval={result['timings']['retrieval_seconds']:.2f}s "
          f"generation={result['timings']['generation_seconds']:.2f}s\n")
    accuracy_latency_results.append({
        "query": q, "answer": result["answer"], "sources": result["sources"],
        "timings": result["timings"], "total_seconds": result["total_seconds"],
        "accuracy_correct": None  # fill in True/False after reading the answer
    })
    time.sleep(13)

[1/10] Who founded Nimbus Robotics?
  Answer: Nimbus Robotics was founded by engineer Priya Kalathil [doc_company_history].
  Total time: 3.58s | retrieval=2.04s generation=1.54s

[2/10] What is the Aster-7's battery life?
  Answer: The Aster-7's battery lasts 14 hours on a single charge [doc_aster7_specs].
  Total time: 3.18s | retrieval=2.08s generation=1.10s

[3/10] How much revenue did Nimbus Robotics report in fiscal year 2033?
  Answer: Nimbus Robotics reported revenue of 340 million rupees in fiscal year 2033 [doc_finance].
  Total time: 3.23s | retrieval=2.01s generation=1.22s

[4/10] Who is Nimbus Robotics' main competitor?
  Answer: Nimbus Robotics' main competitor is Solace Automation [doc_competition].
  Total time: 3.18s | retrieval=2.06s generation=1.12s

[5/10] When is the Aster-8 scheduled for release?
  Answer: The Aster-8 is scheduled for release in early 2035 [doc_aster8_roadmap].
  Total time: 3.22s | retrieval=2.08s generation=1.14s

[6/10] What is the Aster-7's gr

In [6]:
retrieval_times = [r["timings"]["retrieval_seconds"] for r in accuracy_latency_results]
generation_times = [r["timings"]["generation_seconds"] for r in accuracy_latency_results]
total_times = [r["total_seconds"] for r in accuracy_latency_results]

print(f"Average retrieval time: {statistics.mean(retrieval_times):.3f}s")
print(f"Average generation time: {statistics.mean(generation_times):.3f}s")
print(f"Average end-to-end latency: {statistics.mean(total_times):.3f}s")

slowest_component = "generation" if statistics.mean(generation_times) > statistics.mean(retrieval_times) else "retrieval"
print(f"\nSlowest pipeline component: {slowest_component}")

Average retrieval time: 2.032s
Average generation time: 1.176s
Average end-to-end latency: 3.208s

Slowest pipeline component: retrieval


## Dimension 3: Reliability — repeat the same query 10 times, check consistency

In [7]:
reliability_query = "What is the Aster-7's battery life?"
reliability_results = []

for i in range(10):
    print(f"[{i+1}/10] Repeating: {reliability_query}")
    result = generate_answer_timed(reliability_query, _index)
    valid_response = bool(result.get("answer")) and isinstance(result.get("sources"), list)
    reliability_results.append({
        "attempt": i + 1, "answer": result["answer"],
        "valid_response": valid_response, "total_seconds": result["total_seconds"]
    })
    print(f"  Valid: {valid_response} | Answer: {result['answer'][:100]}\n")
    time.sleep(13)

reliability_rate = sum(r["valid_response"] for r in reliability_results) / len(reliability_results) * 100
print(f"Reliability rate: {reliability_rate:.0f}% of repeated calls returned a valid response")

[1/10] Repeating: What is the Aster-7's battery life?
  Valid: True | Answer: The Aster-7's battery lasts 14 hours on a single charge [doc_aster7_specs].

[2/10] Repeating: What is the Aster-7's battery life?
  Valid: True | Answer: The Aster-7's battery lasts 14 hours on a single charge [doc_aster7_specs].

[3/10] Repeating: What is the Aster-7's battery life?
  Valid: True | Answer: The Aster-7's battery lasts 14 hours on a single charge [doc_aster7_specs].

[4/10] Repeating: What is the Aster-7's battery life?
  Valid: True | Answer: The Aster-7's battery lasts 14 hours on a single charge [doc_aster7_specs].

[5/10] Repeating: What is the Aster-7's battery life?
  Valid: True | Answer: The Aster-7's battery lasts 14 hours on a single charge [doc_aster7_specs].

[6/10] Repeating: What is the Aster-7's battery life?
  Valid: True | Answer: The Aster-7's battery lasts 14 hours on a single charge [doc_aster7_specs].

[7/10] Repeating: What is the Aster-7's battery life?
  Valid: True | 

## Dimension 4: Transparency — does every response cite a clear, understandable source?

In [8]:
transparency_results = []
for r in accuracy_latency_results:
    has_sources = len(r["sources"]) > 0
    # a non-technical user should be able to read the title, not just an internal ID
    understandable_titles = all(bool(s.get("title")) for s in r["sources"])
    transparency_results.append({
        "query": r["query"], "has_sources": has_sources,
        "understandable_titles": understandable_titles
    })
    print(f"Query: {r['query']}")
    print(f"  Has sources: {has_sources} | Understandable titles: {understandable_titles}\n")

transparency_rate = sum(t["has_sources"] and t["understandable_titles"] for t in transparency_results) / len(transparency_results) * 100
print(f"Transparency rate: {transparency_rate:.0f}% of responses had clear, understandable citations")

Query: Who founded Nimbus Robotics?
  Has sources: True | Understandable titles: True

Query: What is the Aster-7's battery life?
  Has sources: True | Understandable titles: True

Query: How much revenue did Nimbus Robotics report in fiscal year 2033?
  Has sources: True | Understandable titles: True

Query: Who is Nimbus Robotics' main competitor?
  Has sources: True | Understandable titles: True

Query: When is the Aster-8 scheduled for release?
  Has sources: True | Understandable titles: True

Query: What is the Aster-7's gripper called?
  Has sources: True | Understandable titles: True

Query: How many people does Nimbus Robotics employ?
  Has sources: True | Understandable titles: True

Query: What background does the CTO have?
  Has sources: True | Understandable titles: True

Query: Where has the Aster-7 been deployed?
  Has sources: True | Understandable titles: True

Query: What is Solace Automation's product focus?
  Has sources: True | Understandable titles: True

Transpar

## Dimension 5: Graceful degradation — 5 queries completely outside the knowledge base

In [9]:
out_of_scope_queries = [
    "What is the capital of France?",
    "How do I bake a chocolate cake?",
    "What is Nimbus Robotics' stock ticker symbol?",
    "Who won the last World Cup?",
    "What color is the Aster-7 robot?",
]

degradation_results = []
for i, q in enumerate(out_of_scope_queries, start=1):
    print(f"[{i}/5] {q}")
    result = generate_answer_timed(q, _index)
    honest_refusal = "cannot answer" in result["answer"].lower() or result["low_confidence"]
    print(f"  Answer: {result['answer']}")
    print(f"  Honest refusal / low-confidence flag: {honest_refusal}\n")
    degradation_results.append({
        "query": q, "answer": result["answer"],
        "low_confidence": result["low_confidence"], "honest_refusal": honest_refusal
    })
    time.sleep(13)

degradation_rate = sum(d["honest_refusal"] for d in degradation_results) / len(degradation_results) * 100
print(f"Graceful degradation rate: {degradation_rate:.0f}% of out-of-scope queries were handled honestly")

[1/5] What is the capital of France?
  Answer: I cannot answer this based on the available knowledge base.
  Honest refusal / low-confidence flag: True

[2/5] How do I bake a chocolate cake?
  Answer: I cannot answer this based on the available knowledge base.
  Honest refusal / low-confidence flag: True

[3/5] What is Nimbus Robotics' stock ticker symbol?
  Answer: I cannot answer this based on the available knowledge base.
  Honest refusal / low-confidence flag: True

[4/5] Who won the last World Cup?
  Answer: I cannot answer this based on the available knowledge base.
  Honest refusal / low-confidence flag: True

[5/5] What color is the Aster-7 robot?
  Answer: I cannot answer this based on the available knowledge base.
  Honest refusal / low-confidence flag: True

Graceful degradation rate: 100% of out-of-scope queries were handled honestly


## 10 user-style queries — casual, vague, or verbose phrasing (not engineered test queries)

In [10]:
user_style_queries = [
    "hey so like who even started nimbus robotics lol",
    "battery life???",
    "how much money did they make last year i think it was 2033",
    "who's their biggest rival or whatever",
    "when's the new robot coming out",
    "so um what does the grabby claw thing do exactly",
    "how many ppl work there",
    "tell me abt the boss guy who's the cto",
    "where do they sell their robots at",
    "whats the deal with the other company solace i keep hearing about",
]

user_style_results = []
for i, q in enumerate(user_style_queries, start=1):
    print(f"[{i}/10] \"{q}\"")
    result = generate_answer_timed(q, _index)
    print(f"  Answer: {result['answer']}")
    print(f"  Sources: {[s['source_id'] for s in result['sources']]}")
    print(f"  Low confidence: {result['low_confidence']}\n")
    user_style_results.append({
        "query": q, "answer": result["answer"], "sources": result["sources"],
        "low_confidence": result["low_confidence"],
        "handled_well": None  # fill in True/False after reading the answer
    })
    time.sleep(13)

[1/10] "hey so like who even started nimbus robotics lol"
  Answer: Nimbus Robotics was founded by engineer Priya Kalathil [doc_company_history].
  Sources: ['doc_company_history', 'doc_competition', 'doc_leadership']
  Low confidence: False

[2/10] "battery life???"
  Answer: The Aster-7's battery lasts 14 hours on a single charge [doc_aster7_specs].
  Sources: ['doc_aster7_deployment', 'doc_aster8_roadmap', 'doc_aster7_specs']
  Low confidence: False

[3/10] "how much money did they make last year i think it was 2033"
  Answer: Nimbus Robotics reported revenue of 340 million rupees in fiscal year 2033 [doc_finance].
  Sources: ['doc_finance', 'doc_aster7_deployment', 'doc_aster8_roadmap']
  Low confidence: False

[4/10] "who's their biggest rival or whatever"
  Answer: Nimbus Robotics' main competitor is Solace Automation [doc_competition].
  Sources: ['doc_competition', 'doc_aster7_deployment', 'doc_finance']
  Low confidence: False

[5/10] "when's the new robot coming out"
  Answer

## Product quality report

Fill in this section after reviewing your actual results above.

In [11]:
quality_scores = {
    "accuracy": {
        "score_out_of_5": None,  # fill in based on how many of the 10 accuracy_latency_results were correct
        "notes": "Fill in: how many of the 10 factual queries were answered correctly?"
    },
    "latency": {
        "score_out_of_5": None,  # fill in based on your avg total_seconds
        "notes": "Fill in: what was the average end-to-end latency, and which component was slowest?"
    },
    "reliability": {
        "score_out_of_5": None,  # fill in based on reliability_rate
        "notes": "Fill in: did all 10 repeated calls return valid, well-formed responses?"
    },
    "transparency": {
        "score_out_of_5": None,  # fill in based on transparency_rate
        "notes": "Fill in: did every response include an understandable source citation?"
    },
    "graceful_degradation": {
        "score_out_of_5": None,  # fill in based on degradation_rate
        "notes": "Fill in: did the system honestly refuse all 5 out-of-scope queries?"
    },
}

for dim, data in quality_scores.items():
    print(f"{dim}: {data['score_out_of_5']}/5 — {data['notes']}")

accuracy: None/5 — Fill in: how many of the 10 factual queries were answered correctly?
latency: None/5 — Fill in: what was the average end-to-end latency, and which component was slowest?
reliability: None/5 — Fill in: did all 10 repeated calls return valid, well-formed responses?
transparency: None/5 — Fill in: did every response include an understandable source citation?
graceful_degradation: None/5 — Fill in: did the system honestly refuse all 5 out-of-scope queries?


## Top 3 priorities if this were launching to real users next week

[Fill in after reviewing all your results above. Some likely candidates based on typical
findings across this kind of audit:]

1. **[e.g. Latency]** — if average end-to-end response time is several seconds, that's too
   slow for a chat-like product experience; the generation step is usually the bottleneck,
   not retrieval, since FAISS search over a small index is near-instant.
2. **[e.g. Handling casual phrasing]** — compare how well the system handled the 10
   engineered test queries versus the 10 casual user-style queries; if quality dropped
   noticeably on vague or typo-ridden input, that's a priority since real users don't
   phrase things like a QA engineer.
3. **[e.g. Source citation clarity]** — if citations show only technical IDs (like
   `doc_aster7_specs`) rather than a title a non-technical person could recognize, that's
   a transparency gap worth fixing before real users see it.

Replace this list with your own three priorities based on what your actual test results showed.

In [12]:
# Save all results for the README/report
full_report = {
    "quality_scores": quality_scores,
    "accuracy_latency_results": accuracy_latency_results,
    "reliability_results": reliability_results,
    "transparency_results": transparency_results,
    "degradation_results": degradation_results,
    "user_style_results": user_style_results,
    "average_latency_seconds": statistics.mean(total_times),
    "slowest_component": slowest_component,
}

with open("day21_quality_audit_report.json", "w") as f:
    json.dump(full_report, f, indent=2)

print("Saved full report to day21_quality_audit_report.json")

Saved full report to day21_quality_audit_report.json
